# Belső nyomással terhelt vastag falú henger
Borsodi Bálint TDK-dolgozata, B4 (7.3. táblázat). Negyedmodell, síkalakváltozás. A belső körívet egyenes élek közelítik. A Lamé-megoldással a belső perem csomópontjainak átlagos radiális elmozdulását vetjük össze.

In [1]:
include(joinpath(@__DIR__, "q4_common.jl"))

A nyomásból csomóponti, radiális irányú erők készülnek. Az LLFEM peremintegrálja megadja az egyes csomópontokhoz tartozó húrhossz-súlyokat.

In [2]:
a, b, pi_load = 0.05, 0.12, 50e6
E, ν = 210e9, 0.30
A = pi_load*a^2/(b^2-a^2)
B = A*b^2
uref = (1+ν)/E * ((1-2ν)*A*a + B/a)
cylinder(x,y) = (a + (b-a)*x) .* (cospi(y/2), sinpi(y/2))
errors = zeros(3, 2)
@printf("Lamé uᵣ(a) = %.8e m\n", uref)
println("háló      FI relatív hiba    SRI relatív hiba")
for (i, (nr, nθ)) in enumerate(((2,6), (4,12), (8,24)))
    U, xy = q4_grid(nr, nθ; map=cylinder)
    bc = [BoundaryCondition("bottom"; field=U, uy=0),
          BoundaryCondition("top"; field=U, ux=0)]
    inner = findall(j -> isapprox(norm(xy[:,j]), a), axes(xy,2))
    f = ∫(U ⋅ [1.0, 0.0]; Γ="left")
    for j in inner
        w = f.a[2j-1, 1]
        f.a[2j-1:2j, 1] = pi_load * w * xy[:,j] / a
    end
    for (m, K) in enumerate(q4_stiffness(U, E, ν))
        u = reshape(solveField(K, f; support=bc).a, 2, :)
        ur = mean(dot(u[:,j], xy[:,j]/a) for j in inner)
        errors[i,m] = abs(ur/uref - 1)
    end
    @printf("%dx%-2d     %10.4e         %10.4e\n", nr, nθ, errors[i,:]...)
end
@assert all(diff(errors; dims=1) .< 0) && all(errors[:,2] .< errors[:,1])
ref = [0.0583 0.0213; 0.0174 0.00624; 0.00461 0.00164]
@assert all(isapprox.(errors, ref; rtol=0.005))

Lamé uᵣ(a) = 2.00280112e-05 m
háló      FI relatív hiba    SRI relatív hiba
2x6      5.8345e-02         2.1316e-02
4x12     1.7438e-02         6.2368e-03
8x24     4.6069e-03         1.6366e-03
